# 03 · 广播机制 Broadcasting

> **能力标签**：SH3（NumPy 与向量化计算 / NumPy & vectorized computing）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 NumPy **广播规则（broadcasting rules）** 预判两个数组能否进行运算。
2. 理解"维度对齐从右到左"的原则，正确处理形状不匹配问题。
3. 实现逐列标准化（`standardize_columns`），理解 `axis=0` 与 `keepdims` 的作用。
4. 区分广播与显式循环的代码风格与性能差异。

> 对应能力：**SH3**


## 概念讲解 Concepts

### 广播规则 Broadcasting Rules

当两个数组形状不同时，NumPy 尝试**广播（broadcast）**它们，使其形状兼容再运算。
规则（从右对齐维度，逐维比较）：

1. 若维度数不同，**在形状左侧补 1**（`(3,)` → `(1, 3)`，`(4,)` → `(1, 4)`）。
2. 对每个维度：若两者相等，OK；若其中一个为 1，**沿该维度重复（stretch）**；否则报错。

```
A: (4, 3)    B: (3,)   →  B 自动左补 1 → (1, 3) → stretch → (4, 3)  ✓
A: (4, 3)    B: (4,)   →  B 自动左补 1 → (1, 4) → 末位 3 ≠ 4 且都不为 1 → ValueError ✗
                           ※ 若需逐列广播，需显式 reshape：B.reshape(4, 1) → (4, 1)
                           → (4, 3) 与 (4, 1) 广播：末位 3 vs 1 → stretch → (4, 3)  ✓（需显式）
A: (4, 3)    B: (4, 2) →  报错（3 ≠ 2 且都不为 1）                               ✗
```

> **常见错误**：误以为 `(4,)` 会右补变成 `(4, 1)`——实际上 NumPy **总是在左侧补 1**。
> 需要 `(4, 1)` 形状时，请显式写 `arr.reshape(4, 1)` 或 `arr[:, np.newaxis]`。

---

### 按列操作 Column-wise Operations

矩阵 $X$ 形状 $(n, p)$，对每列计算均值/标准差：

```python
mu  = X.mean(axis=0)    # shape: (p,)  — 沿 axis=0（行方向）折叠
std = X.std(axis=0)     # shape: (p,)
Z   = (X - mu) / std    # 广播：(n,p) - (p,) → 自动对齐最后一维
```

---

### keepdims

`keepdims=True` 保留被折叠的维度（以 1 填充），使广播更直观：

```python
X.mean(axis=0, keepdims=True)   # shape: (1, p)
X.mean(axis=1, keepdims=True)   # shape: (n, 1)  ← 需要时用这个
```

---

### 实现逐列 z-score

```python
def standardize_columns(X):
    X = np.asarray(X, dtype=float)
    return (X - X.mean(axis=0)) / X.std(axis=0)
```

等价于对每列单独做 z-score，再拼接——但广播版一行代码，无循环。

## 示例 Worked Example：逐列标准化（standardize_columns）

In [ ]:
import numpy as np

# ── 数据：5 行 3 列 ────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
X = rng.normal(loc=[10.0, 100.0, 0.5], scale=[1.0, 10.0, 0.1], size=(5, 3))
print("原始矩阵 X (shape={}):\n{}".format(X.shape, X.round(3)))

# ── 逐列均值与标准差 ─────────────────────────────────────────────────────
mu  = X.mean(axis=0)   # shape: (3,)
std = X.std(axis=0)    # shape: (3,)
print("\n列均值:", mu.round(3))
print("列标准差:", std.round(3))

# ── 广播减法与除法 ────────────────────────────────────────────────────────
# (5,3) - (3,) → OK，右对齐后 (3,) 相当于 (1,3) → stretch → (5,3)
Z = (X - mu) / std

print("\n标准化后 Z (shape={}):\n{}".format(Z.shape, Z.round(4)))
print("\nZ 各列均值 (应≈0):", Z.mean(axis=0).round(10))
print("Z 各列标准差 (应≈1):", Z.std(axis=0).round(10))


In [ ]:
# 广播形状演示 —— 理解规则
import numpy as np

a = np.ones((4, 3))
b = np.array([1, 2, 3])         # shape: (3,)
c = np.array([[10], [20], [30], [40]])  # shape: (4, 1)

print("a.shape:", a.shape)
print("b.shape:", b.shape, "→ 广播为 (1,3) → stretch → (4,3)")
print("(a + b).shape:", (a + b).shape)

print("\nc.shape:", c.shape, "→ 广播为 (4,1) → stretch → (4,3)")
print("(a + c).shape:", (a + c).shape)

# 行均值归一化（axis=1 广播）
X = np.arange(12, dtype=float).reshape(3, 4)
row_mean = X.mean(axis=1, keepdims=True)   # shape: (3,1)
X_centered = X - row_mean                  # (3,4) - (3,1) → OK
print("\nX 行中心化:\n", X_centered)


## 动手 Exercise

实现 `row_softmax_broadcast`：对矩阵每行做 softmax（数值稳定版）。
要求：只用广播，不使用循环。

$$\text{softmax}(x_i) = \frac{e^{x_i - \max_j x_j}}{\sum_j e^{x_j - \max_j x_j}}$$


In [ ]:
import numpy as np

def row_softmax_broadcast(X: np.ndarray) -> np.ndarray:
    """逐行 softmax（数值稳定，broadcasting 实现）。"""
    X = np.asarray(X, dtype=float)
    # 减去每行最大值（防止 exp 溢出）
    m = X.max(axis=1, keepdims=True)   # shape: (n, 1)
    e = np.exp(X - m)                  # broadcast: (n,p) - (n,1)
    return e / e.sum(axis=1, keepdims=True)

# 验证
rng = np.random.default_rng(3)
X = rng.normal(size=(4, 5))
S = row_softmax_broadcast(X)
print("输出 shape:", S.shape)
print("每行之和 (应全为 1):", S.sum(axis=1).round(10))
print("所有值 > 0:", (S > 0).all())


## 延伸阅读 Further Reading

1. **NumPy 官方文档 — Broadcasting**: <https://numpy.org/doc/stable/user/basics.broadcasting.html>
2. **可视化工具**: <https://numpy.org/doc/stable/user/theory.broadcasting.html>
3. **《Python Data Science Handbook》— Broadcasting Arrays** — Jake VanderPlas
4. **Scipy Lectures — Broadcasting**: <https://scipy-lectures.org/intro/numpy/operations.html#broadcasting>


## 关联练习 Related Assignments

本课对应练习包：**`w2-broadcasting`**（任务：`standardize_columns`）

```bash
python check.py w2-broadcasting
```

> 能力标签：**SH3** · threshold ≥ 0.7
